<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/check_annotation_result_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

In [ ]:
import pandas as pd

# Define file paths
base_dir = '/content/drive/MyDrive/world-inflation/data/reddit/production/'
path_annotation = base_dir + 'annotation_result.tsv'
path_test_200 = base_dir + 'test-data-200.csv'
path_prod_1039 = base_dir + '2-main-prod-1039.csv'

# Load the TSV file (annotation_result.tsv)
# Assuming it is tab-separated
df_annotation = pd.read_csv(path_annotation, sep='\t')

# Load the CSV files
# standard read_csv handles quoted strings automatically
df_test_200 = pd.read_csv(path_test_200)
df_prod_1039 = pd.read_csv(path_prod_1039)

print(f"Loaded annotation_result: {len(df_annotation)} records")
print(f"Loaded test-data-200: {len(df_test_200)} records")
print(f"Loaded 2-main-prod-1039: {len(df_prod_1039)} records")

In [ ]:
# Create a set of bodies to exclude for faster lookup
# Combining bodies from both CSV files
exclude_bodies = set(df_test_200['body']).union(set(df_prod_1039['body']))

# Filter annotation_result to find records NOT in the exclusion set
# using boolean indexing with ~ (not) and isin()
missing_records = df_annotation[~df_annotation['body'].isin(exclude_bodies)]

# Output the results
print(f"Number of missing records found: {len(missing_records)}")
print("\n--- Missing Records ---")

# Iterate and print each found record
for index, row in missing_records.iterrows():
    print(f"Index: {index}")
    print(f"Body: {row['body']}")
    # Print other columns if needed
    print(f"Subreddit ID: {row.get('subreddit_id', 'N/A')}")
    print("-" * 30)

# If you want to see the dataframe directly:
# missing_records

In [ ]:
# Check for duplicates based on the 'body' column
# keep=False marks all instances of duplicates as True (so you can see all copies)
duplicate_records = df_annotation[df_annotation.duplicated(subset=['body'], keep=False)]

print(f"Total duplicate records found: {len(duplicate_records)}")

if len(duplicate_records) > 0:
    print("\n--- List of Duplicate Records ---")

    # Sort by body so duplicates appear next to each other
    duplicate_records_sorted = duplicate_records.sort_values(by='body')

    for index, row in duplicate_records_sorted.iterrows():
        print(f"Index: {index}")
        # Print subreddit_id to see if they come from different sources
        print(f"Subreddit ID: {row.get('subreddit_id', 'N/A')}")
        print(f"Body: {row['body']}")
        print("-" * 30)
else:
    print("No duplicate records found based on 'body' column.")